# Project 3 — Messy-text structured extractor
**Track A (local Ollama).** Turn messy text into validated JSON.
**Data:** `data/job_posts.jsonl` — 18 posts labelled with `title`, `seniority`, `remote`.
**Evaluated on:** field-level accuracy vs the labels.

In [ ]:
import sys, json; sys.path.append("../..")   # import utils/ and eval/ from repo root
from pydantic import BaseModel, ValidationError
from utils import ask

POSTS = [json.loads(l) for l in open("data/job_posts.jsonl", encoding="utf-8")]
print(len(POSTS), "posts — example:", POSTS[0])

## Starter: schema + validate + repair

In [ ]:
class JobPost(BaseModel):
    title: str
    seniority: str        # junior | mid | senior
    remote: bool

def extract(text, retries=1):
    prompt = ('Return ONLY JSON: {"title": str, "seniority": "junior|mid|senior", "remote": bool}\n\n'
              f"Text:\n{text}")
    raw = ask(prompt)
    for _ in range(retries + 1):
        try:
            return JobPost(**json.loads(raw))
        except (json.JSONDecodeError, ValidationError) as e:
            raw = ask(prompt + f"\n\nYour last output was invalid ({e}). Return valid JSON only.")
    raise ValueError("no valid JSON: " + raw[:120])

print(extract(POSTS[0]["text"]))

## Field-level scoring over the dataset

In [ ]:
def field_scores(posts):
    fields = ["title", "seniority", "remote"]
    hits = {f: 0 for f in fields}
    for p in posts:
        try:
            got = extract(p["text"]).model_dump()
        except Exception:
            continue
        for f in fields:
            if str(got.get(f)).lower() == str(p[f]).lower():
                hits[f] += 1
    n = len(posts)
    for f in fields:
        print(f"{f:10s} {hits[f]}/{n} = {hits[f]/n:.0%}")

field_scores(POSTS[:6])   # start small; run all 18 once it works

## Your tasks
1. Improve the prompt so `title` normalises consistently.
2. Run the full 18 and report per-field accuracy.
3. Force a prose reply and confirm the repair step recovers.

In [ ]:
# TODO: your code here
